In [0]:
from pyspark.sql.functions import col

# =========================================================
# STEP-5 : GET RAW INVALID RECORDS FROM QUARANTINE
# =========================================================

invalid_df = spark.read.table("retails.silver.orders_quarantine") \
                    .filter((col("_rescued_data").isNotNull()) & (col("quarantine_status") == 'NEW'))

In [0]:
from pyspark.sql.functions import col, get_json_object as get_json_from

# =========================================================
# STEP-6 : TRY TO RECOVER RESCUED COLUMNS
# =========================================================
recovered_df = invalid_df \
                .withColumn("order_id_fixed", get_json_from(col("_rescued_data"), "$.order_id")) \
                .withColumn("order_date_fixed", get_json_from(col("_rescued_data"), "$.order_date")) \
                .withColumn("order_customer_id_fixed", get_json_from(col("_rescued_data"), "$.order_customer_id")) \
                .withColumn("order_status_fixed", get_json_from(col("_rescued_data"), "$.order_status"))


In [0]:
from pyspark.sql.functions import coalesce, to_date

# =========================================================
# STEP-7 : MERGE RECOVERED VALUES
# =========================================================

recovered_df = recovered_df \
    .withColumn(
        "order_id",
        coalesce(col("order_id"), col("order_id_fixed").cast("bigint"))
    ) \
    .withColumn(
        "order_date",
        coalesce(col("order_date"), to_date(col("order_date_fixed"), "yyyy-MM-dd"))
    ) \
    .withColumn(
        "order_customer_id",
        coalesce(col("order_customer_id"), col("order_customer_id_fixed").cast("bigint"))
    ) \
    .withColumn(
        "order_status",
        coalesce(col("order_status"), col("order_status_fixed").cast("string"))
    )

In [0]:
recovered_df = recovered_df.drop("order_id_fixed", "order_date_fixed", "order_customer_id_fixed", "order_status_fixed")

In [0]:
from pyspark.sql.functions import col

# =========================================================
# STEP-8 : APPLY DATA QUALITY RULES
# =========================================================

cleaned_recovered_df = recovered_df.filter(
    col("order_id").isNotNull() &
    col("order_date").isNotNull() &
    col("order_customer_id").isNotNull() &
    col("order_status").isNotNull()
)

In [0]:
cleaned_recovered_df = cleaned_recovered_df.dropDuplicates(["order_id"])
cleaned_recovered_df.createOrReplaceTempView("orders_cleaned_vw_fixed")

In [0]:
from pyspark.sql.functions import when, current_timestamp, sha2, concat_ws

cleaned_recovered_df = cleaned_recovered_df \
    .withColumn("order_id", col("order_id").cast("bigint")) \
    .withColumn("order_date", col("order_date").cast("date")) \
    .withColumn("order_customer_id", col("order_customer_id").cast("bigint")) \
    .withColumn("batch_id", col("batch_id").cast("integer")) \
    .withColumn("is_deleted", when(col("op")=='DELETE', True).otherwise(False))



cleaned_recovered_df = cleaned_recovered_df.select("order_id", "order_date", "order_customer_id", "order_status", "op", "is_deleted", "source_system", "source_file_name", "ingestion_ts", "ingestion_dt", "batch_id", "run_id")

cleaned_recovered_df = cleaned_recovered_df.withColumn("event_ts", current_timestamp()) \
                    .withColumn("record_hash",
                                sha2(
                                    concat_ws(
                                        "||",
                                        col("order_id"),
                                        col("order_date"),
                                        col("order_customer_id"),
                                        col("order_status")
                                    ),
                                    256
                                )
                            )
    

try:
    cleaned_recovered_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "false") \
        .saveAsTable("retails.silver.orders_cdc")

except Exception as e:
    print(str(e))

In [0]:
merge_query_rescued = """
    MERGE INTO retails.silver.orders_quarantine t
    USING orders_cleaned_vw_fixed s
    ON t.order_id = s.order_id
    WHEN MATCHED THEN
        UPDATE SET t.quarantine_status = 'FIXED', t.reprocessed_at = current_timestamp()
 
    """
spark.sql(merge_query_rescued).show()


In [0]:
update_query_corrupt = """
        UPDATE retails.silver.orders_quarantine
        SET
            quarantine_status = 'INVALID',
            reprocessed_at = current_timestamp()
        WHERE quarantine_status = 'NEW'
"""

spark.sql(update_query_corrupt).show()

In [0]:
%sql
-- select * from retails.silver.orders_quarantine;
-- select * from retails.silver.orders_cdc;


In [0]:
# %sql
# ALTER TABLE retails.silver.orders_quarantine
# ADD COLUMNS (
#     reprocessed_at TIMESTAMP
# )